<a href="https://colab.research.google.com/github/dgaida/wpf_cps_th_public/blob/main/assets/exercises/week_07/03_lineares_system_1d_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1D Lineares Kinematisches System: Zustandsraummodellierung & Simulation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/wpf_cps_th_public/blob/main/assets/exercises/week_07/03_lineares_system_1d_simulation.ipynb)

In diesem Notebook modellieren und simulieren wir ein **1-dimensionales lineares dynamisches System** eines sich bewegenden Objekts (hier visualisiert als ein **Fahrrad**).

### Physikalische Zustandsgrößen:
Der Zustand zum diskreten Zeitschritt $t_k$ wird durch den Zustandsvektor $x_k \in \mathbb{R}^3$ beschrieben:

$$
x_k = \begin{bmatrix} p_k \\ v_k \\ a_k \end{bmatrix}
$$

Dabei bezeichnen:
- $p_k$: Position zum Zeitpunkt $t_k$ (in $\text{m}$)
- $v_k$: Geschwindigkeit zum Zeitpunkt $t_k$ (in $\text{m/s}$)
- $a_k$: Beschleunigung zum Zeitpunkt $t_k$ (in $\text{m/s}^2$)

### Zeitdiskretisierung:
Der zeitliche Abstand zwischen zwei aufeinanderfolgenden Zeitschritten $t_k$ und $t_{k+1}$ ist fest vorgegeben:

$$
\Delta t = t_{k+1} - t_k = 1.0\,\text{s}
$$

### Kinematische Bewegungsgleichungen:
Bei einer konstanten Beschleunigung $a_k$ über das Intervall $[t_k, t_{k+1})$ gilt:

$$
p_{k+1} = p_k + v_k \cdot \Delta t + \frac{1}{2} a_k \cdot \Delta t^2
$$

$$
v_{k+1} = v_k + a_k \cdot \Delta t
$$

In vektorieller Matrix-Schreibweise ergibt sich das lineare Zustandsraummodell:

$$
x_{k+1} = A \cdot x_k
$$

mit der Systemmatrix $A \in \mathbb{R}^{3 \times 3}$:

$$
A = \begin{bmatrix}
1 & \Delta t & \frac{1}{2}\Delta t^2 \\
0 & 1 & \Delta t \\
0 & 0 & 1
\end{bmatrix}
$$


## 1. Benötigte Bibliotheken importieren

Wir importieren `numpy` für Matrix- und Vektorrechnungen, `matplotlib.pyplot` für die Plots und `matplotlib.animation` zur Erstellung der Fahrrad-Animation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle
from IPython.display import HTML

# Inline-Darstellung konfigurieren
%matplotlib inline
plt.rcParams['font.size'] = 11


## 2. Aufgabe: Programmierung der Systemmatrix $A$

Definieren Sie das Zeitintervall $\Delta t = 1.0\,\text{s}$ und programmieren Sie die Systemmatrix $A \in \mathbb{R}^{3 \times 3}$ als zweidimensionales NumPy-Array (`np.ndarray`).

Nutzen Sie dazu die exakte kinematische Zustandsübergangsmatrix:
$$
A = \begin{bmatrix}
1 & \Delta t & \frac{1}{2}\Delta t^2 \\
0 & 1 & \Delta t \\
0 & 0 & 1
\end{bmatrix}
$$

In [ ]:
# Zeitabstand Delta t zwischen t_{k+1} und t_k in Sekunden
dt = 1.0

# TODO: Programmieren Sie hier die Systemmatrix A als 3x3 NumPy-Array
# A = np.array([
#     [..., ..., ...],
#     [..., ..., ...],
#     [..., ..., ...]
# ], dtype=float)

A = None  # Ersetzen Sie None durch Ihre Matrix-Definition

print("Systemmatrix A:")
print(A)


### Validierung der Systemmatrix $A$

Führen Sie die nachfolgende Zelle aus, um zu prüfen, ob die Matrix $A$ die korrekte Dimension und Werte besitzt.

In [ ]:
if A is None:
    raise ValueError("Bitte definieren Sie die Systemmatrix A in der vorherigen Zelle!")

assert isinstance(A, np.ndarray), "A muss ein NumPy-Array sein!"
assert A.shape == (3, 3), f"A muss Dimension (3, 3) haben, hat aber {A.shape}."

A_expected = np.array([[1.0, dt, 0.5 * dt**2], [0.0, 1.0, dt], [0.0, 0.0, 1.0]])
np.testing.assert_allclose(A, A_expected, err_msg="Werte der Systemmatrix A stimmen nicht mit den theoretischen Kinematikgleichungen überein!")
print("✓ Sehr gut! Die Systemmatrix A ist korrekt implementiert.")


## 3. Vorgabe der Beschleunigung $a_k$ als stückweise diskrete Funktion

Die Studierenden können die Beschleunigung als stückweise diskrete Funktion vorgeben. Bspw. als Vektor, wobei jeder Wert einer Beschleunigung zum diskreten Zeitschritt $t_k$ entspricht:

$$
a = [a_0, a_1, a_2, \dots, a_{N-1}]
$$

Ein typisches Fahrszenario für ein Fahrrad:
1. **Anfahren**: Positive Beschleunigung ($a_k > 0$)
2. **Gleichförmige Fahrt**: Geschwindigkeit wird gehalten ($a_k = 0$)
3. **Bremsen**: Negative Beschleunigung ($a_k < 0$) bis zum Stillstand ($v_k = 0$)
4. **Halt / Stillstand**: $a_k = 0$

In [ ]:
# Beispielhafter Beschleunigungsvektor: jeder Wert entspricht der Beschleunigung zum Zeitpunkt t_k
# Dauer je Schritt ist Delta t = 1.0 s
# Sie können dieses Profil beliebig anpassen oder eigene Beschleunigungswerte testen!
a_k_profile = np.array([
    0.0,        # t = 0 s: Start im Stillstand
    1.0, 1.0,   # t = 1..2 s: Beschleunigungsphase (1.0 m/s^2)
    0.5, 0.5,   # t = 3..4 s: Sanftes Weiterbeschleunigen (0.5 m/s^2)
    0.0, 0.0,   # t = 5..6 s: Konstante Fahrt ohne Beschleunigung
    0.0, 0.0,   # t = 7..8 s: Weiterhin gleichförmig
    -1.0, -1.0, # t = 9..10 s: Bremsphase (-1.0 m/s^2)
    -1.0,       # t = 11 s: Letzter Bremsschritt zum Stillstand
    0.0, 0.0    # t = 12..13 s: Fahrrad steht
], dtype=float)

print(f"Anzahl der Zeitschritte: {len(a_k_profile)}")
print(f"Simulationsdauer: {len(a_k_profile) * dt:.1f} s")


## 4. Simulation des Systems mit $x_{k+1} = A \cdot x_k$

Wir berechnen nun schrittweise den zeitlichen Verlauf des Zustandsvektors:
$$
x_k = \begin{bmatrix} p_k \\ v_k \\ a_k \end{bmatrix}
$$

Zum Zeitpunkt $t_k$ wird die Beschleunigung $a_k$ auf den vorgegebenen Vektorwert gesetzt. Anschließend wird der Folgezustand $x_{k+1}$ durch Multiplikation mit der Systemmatrix $A$ berechnet:
$$
x_{k+1} = A \cdot x_k
$$


In [ ]:
def simulate_1d_system(A: np.ndarray, a_profile: np.ndarray, dt: float = 1.0, p_0: float = 0.0, v_0: float = 0.0):
    """Simuliert das 1D lineare kinematische System über die Zeit.
    
    Args:
        A: 3x3 Systemmatrix.
        a_profile: 1D-Array mit diskreten Beschleunigungswerten für jeden Zeitschritt k.
        dt: Zeitschrittweite in Sekunden (Standard: 1.0 s).
        p_0: Anfangsposition in Metern.
        v_0: Anfangsgeschwindigkeit in m/s.
        
    Returns:
        tuple: (time_steps, p_k, v_k, a_k)
    """
    num_steps = len(a_profile)
    
    # Historie der Zustände initialisieren (inklusive Endzustand nach num_steps)
    times = np.zeros(num_steps + 1)
    p_k = np.zeros(num_steps + 1)
    v_k = np.zeros(num_steps + 1)
    a_k = np.zeros(num_steps + 1)
    
    # Anfangszustand bei t = 0
    current_state = np.array([p_0, v_0, a_profile[0]], dtype=float)
    p_k[0] = current_state[0]
    v_k[0] = current_state[1]
    a_k[0] = current_state[2]
    times[0] = 0.0
    
    # Simulationsschleife
    for k in range(num_steps):
        # Aktuelle Beschleunigung zum Zeitschritt k zuweisen
        current_state[2] = a_profile[k]
        a_k[k] = current_state[2]
        
        # Zustandstransition über die Systemmatrix A: x_{k+1} = A * x_k
        next_state = A @ current_state
        
        # Position und Geschwindigkeit für k+1 übernehmen
        p_k[k + 1] = next_state[0]
        v_k[k + 1] = next_state[1]
        
        # Beschleunigung für Zeitschritt k+1 (sofern innerhalb des Profils)
        if k + 1 < num_steps:
            current_state = np.array([next_state[0], next_state[1], a_profile[k + 1]], dtype=float)
            a_k[k + 1] = a_profile[k + 1]
        else:
            current_state = np.array([next_state[0], next_state[1], 0.0], dtype=float)
            a_k[k + 1] = 0.0
            
        times[k + 1] = (k + 1) * dt
        
    return times, p_k, v_k, a_k

# Simulation ausführen
times, p_k, v_k, a_k = simulate_1d_system(A, a_k_profile, dt=dt, p_0=0.0, v_0=0.0)

print(f"Endposition:       p_end = {p_k[-1]:.2f} m")
print(f"Endgeschwindigkeit: v_end = {v_k[-1]:.2f} m/s")
print(f"Maximale Geschw.:  v_max = {np.max(v_k):.2f} m/s ({np.max(v_k) * 3.6:.1f} km/h)")


## 5. Zeitverläufe von Position $p_k$, Geschwindigkeit $v_k$ und Beschleunigung $a_k$

Wir plotten nun die sich ergebende Position, Geschwindigkeit und Beschleunigung über der Zeit.

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(11, 8), sharex=True)

# 1. Plot: Position p_k über der Zeit
axs[0].plot(times, p_k, 'b-o', markersize=5, linewidth=2, label=r'Position $p_k$ [m]')
axs[0].set_ylabel('Position $p_k$ [m]', fontsize=12)
axs[0].set_title(r'1D Lineares System: Zustandsverläufe über der Zeit ($\Delta t = 1.0\,\mathrm{s}$)', fontsize=14, fontweight='bold')
axs[0].grid(True, linestyle='--', alpha=0.7)
axs[0].legend(loc='upper left')

# 2. Plot: Geschwindigkeit v_k über der Zeit
axs[1].plot(times, v_k, 'g-s', markersize=5, linewidth=2, label=r'Geschwindigkeit $v_k$ [m/s]')
axs[1].set_ylabel('Geschwindigkeit $v_k$ [m/s]', fontsize=12)
axs[1].grid(True, linestyle='--', alpha=0.7)
axs[1].legend(loc='upper left')

# 3. Plot: Beschleunigung a_k über der Zeit (als stufenförmige / stückweise diskrete Funktion)
axs[2].step(times[:-1], a_k_profile, where='post', color='crimson', linewidth=2, label=r'Vorgegebene Beschleunigung $a_k$ [m/s²]')
axs[2].scatter(times[:-1], a_k_profile, color='crimson', s=35, zorder=3)
axs[2].set_ylabel('Beschleunigung $a_k$ [m/s²]', fontsize=12)
axs[2].set_xlabel('Zeit $t$ [s]', fontsize=12)
axs[2].grid(True, linestyle='--', alpha=0.7)
axs[2].legend(loc='upper left')

plt.tight_layout()
plt.show()


## 6. Fahrrad-Animation in der 1D-Umgebung

Unter dem Plot visualisieren wir nun das **Fahrrad**, welches mit der simulierten Position $p_k$ und Geschwindigkeit $v_k$ von **links nach rechts** über die 1D-Fahrbahn fährt.

Das Fahrrad wird detailgetreu mit Rahmen, rotierenden Laufrädern, Lenker und Sattel animiert. Die Position des Fahrrads auf der X-Achse entspricht exakt der berechneten Position $p_k$.

In [ ]:
# Erstellung der Animation des Fahrrads entlang der 1D-Achse
fig_anim, ax_anim = plt.subplots(figsize=(12, 3.8))

p_min = min(np.min(p_k), 0.0) - 2.0
p_max = max(np.max(p_k), 10.0) + 5.0

ax_anim.set_xlim(p_min, p_max)
ax_anim.set_ylim(-0.3, 3.8)
ax_anim.set_aspect('equal')
ax_anim.set_xlabel('1D Position $p$ [m]', fontsize=12)
ax_anim.set_title('Fahrrad-Fahrt entlang der 1D-Umgebung', fontsize=13, fontweight='bold')
ax_anim.grid(True, linestyle=':', alpha=0.6)

# Fahrbahn und Markierungen
ax_anim.axhline(0.0, color='dimgray', linewidth=3.5, label='Fahrbahn')
ticks_x = np.arange(int(np.floor(p_min)), int(np.ceil(p_max)) + 1, 5)
for tx in ticks_x:
    if p_min <= tx <= p_max:
        ax_anim.plot([tx, tx], [0.0, -0.2], color='dimgray', linewidth=1.5)
        ax_anim.text(tx, -0.3, f"{tx}m", ha='center', va='top', fontsize=9, color='dimgray')

# Fahrrad-Geometrie
r_wheel = 0.6          # Laufrad-Radius [m]
wheelbase_half = 1.0   # Halber Radstand [m]

# Patches für Laufräder
rear_wheel = Circle((0, r_wheel), r_wheel, color='#222222', fill=False, lw=3)
front_wheel = Circle((0, r_wheel), r_wheel, color='#222222', fill=False, lw=3)
rear_hub = Circle((0, r_wheel), 0.08, color='black', fill=True)
front_hub = Circle((0, r_wheel), 0.08, color='black', fill=True)

# Speichen (rotieren mit Drehwinkel)
rear_spokes, = ax_anim.plot([], [], color='gray', lw=1.2)
front_spokes, = ax_anim.plot([], [], color='gray', lw=1.2)

# Fahrradrahmen, Gabel, Sattel und Lenker
frame_line, = ax_anim.plot([], [], color='#0055bb', lw=3.5, solid_capstyle='round')
fork_line, = ax_anim.plot([], [], color='#003388', lw=3.0, solid_capstyle='round')
saddle_line, = ax_anim.plot([], [], color='black', lw=4.5, solid_capstyle='round')
handlebar_line, = ax_anim.plot([], [], color='black', lw=3.5, solid_capstyle='round')

# Status-Text für Zeit, Position und Geschwindigkeit
status_text = ax_anim.text(
    0.02, 0.88, '',
    transform=ax_anim.transAxes,
    fontsize=11,
    fontfamily='monospace',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='navy', alpha=0.9)
)

ax_anim.add_patch(rear_wheel)
ax_anim.add_patch(front_wheel)
ax_anim.add_patch(rear_hub)
ax_anim.add_patch(front_hub)

# Interpolation für flüssige Animation (z.B. 10 Zwischenschritte pro Sekunde)
fps = 15
t_fine = np.linspace(times[0], times[-1], int((times[-1] - times[0]) * fps) + 1)
p_fine = np.interp(t_fine, times, p_k)
v_fine = np.interp(t_fine, times, v_k)

def init_anim():
    frame_line.set_data([], [])
    fork_line.set_data([], [])
    saddle_line.set_data([], [])
    handlebar_line.set_data([], [])
    rear_spokes.set_data([], [])
    front_spokes.set_data([], [])
    status_text.set_text('')
    return rear_wheel, front_wheel, rear_hub, front_hub, frame_line, fork_line, saddle_line, handlebar_line, rear_spokes, front_spokes, status_text

def update_anim(frame_idx):
    xc = p_fine[frame_idx]
    current_t = t_fine[frame_idx]
    current_v = v_fine[frame_idx]
    
    yw = r_wheel
    xr = xc - wheelbase_half
    xf = xc + wheelbase_half
    
    # Positionierung der Räder
    rear_wheel.center = (xr, yw)
    front_wheel.center = (xf, yw)
    rear_hub.center = (xr, yw)
    front_hub.center = (xf, yw)
    
    # Speichenrotation basierend auf zurückgelegter Distanz (Raddrehung: theta = -dist / r)
    theta = -xc / r_wheel
    angles = np.array([theta, theta + np.pi/2, theta + np.pi/4, theta + 3*np.pi/4])
    
    spokes_r_x = []
    spokes_r_y = []
    spokes_f_x = []
    spokes_f_y = []
    for ang in angles:
        spokes_r_x.extend([xr - r_wheel * np.cos(ang), xr + r_wheel * np.cos(ang), np.nan])
        spokes_r_y.extend([yw - r_wheel * np.sin(ang), yw + r_wheel * np.sin(ang), np.nan])
        spokes_f_x.extend([xf - r_wheel * np.cos(ang), xf + r_wheel * np.cos(ang), np.nan])
        spokes_f_y.extend([yw - r_wheel * np.sin(ang), yw + r_wheel * np.sin(ang), np.nan])
        
    rear_spokes.set_data(spokes_r_x, spokes_r_y)
    front_spokes.set_data(spokes_f_x, spokes_f_y)
    
    # Rahmenpunkte
    xb = xc - 0.2        # Tretlager (Bottom Bracket)
    yb = yw
    xs = xc - 0.5        # Sattelrohr-Knotenpunkt
    ys = yw + 1.05
    xh = xc + 0.65       # Steuerrohr (Head tube)
    yh = yw + 1.15
    
    # Diamantrahmen
    frame_x = [xr, xb, xs, xr, xb, xh, xs]
    frame_y = [yw, yb, ys, yw, yb, yh, ys]
    frame_line.set_data(frame_x, frame_y)
    
    # Gabel und Lenker
    fork_x = [xf, xh, xh - 0.08, xh + 0.15]
    fork_y = [yw, yh, yh + 0.22, yh + 0.22]
    fork_line.set_data(fork_x, fork_y)
    
    # Lenkergriff
    handlebar_line.set_data([xh + 0.05, xh + 0.2], [yh + 0.22, yh + 0.20])
    
    # Sattel
    saddle_line.set_data([xs - 0.22, xs + 0.18], [ys + 0.12, ys + 0.12])
    
    # Status-Anzeige
    status_text.set_text(f"Zeit t: {current_t:4.1f} s | Position p: {xc:5.2f} m | Geschw. v: {current_v:4.2f} m/s ({current_v*3.6:4.1f} km/h)")
    
    return rear_wheel, front_wheel, rear_hub, front_hub, frame_line, fork_line, saddle_line, handlebar_line, rear_spokes, front_spokes, status_text

anim = animation.FuncAnimation(
    fig_anim,
    update_anim,
    frames=len(t_fine),
    init_func=init_anim,
    interval=1000 / fps,
    blit=True
)

plt.close(fig_anim)

# Animation im Notebook als interaktives JavaScript-HTML anzeigen
HTML(anim.to_jshtml())


## 7. Zusammenfassung

- **Zustandsraummodell**: Das 1D-System wird durch $x_k = [p_k, v_k, a_k]^T$ beschrieben.
- **Systemmatrix $A$**:
  $$
  A = \begin{bmatrix}
  1 & \Delta t & \frac{1}{2}\Delta t^2 \\
  0 & 1 & \Delta t \\
  0 & 0 & 1
  \end{bmatrix}
  $$
  beschreibt die deterministische Zustandstransition für diskrete Zeitschritte mit Schrittweite $\Delta t = 1.0\,\text{s}$.
- **Stückweise diskrete Beschleunigung**: Beliebige diskrete Beschleunigungsverläufe $a_k$ können als Input vorgegeben werden, woraus sich durch numerische Simulation die Trajektorien von Geschwindigkeit $v_k$ und Position $p_k$ ergeben.
- **Visualisierung**: Mit `matplotlib` und `FuncAnimation` kann die physikalische Bewegung des Fahrrads direkt visuell überprüft und nachvollzogen werden.
